# Глава 5: Ваша первая программа на Python
**Создайте свою первую большую языковую модель (LLM) — Глава 5: Ваша первая программа на Python**

Этот блокнот содержит исполняемые примеры кода из Главы 5. Запускайте ячейки сверху вниз.

- Установки: transformers (для демо GPT-2)
- Данные: небольшие встроенные строки; внешние файлы не требуются
- Достаточно CPU. При первом запуске установка библиотек и загрузка весов GPT-2 (около 500 МБ) могут занять несколько минут.

## Настройка

In [ ]:
# ===== УСТАНОВКА =====
# Установить библиотеку transformers (набор инструментов HuggingFace для работы с LLM)
!pip install -q transformers==5.23.0 torch==2.15.0  # проверенная пара (Приложение A)

import warnings
warnings.filterwarnings('ignore')  # Отключить незначительные предупреждения о версиях

# Загрузить инструменты HuggingFace для GPT-2
# "pipeline" это вспомогательная функция, которая загружает модель и управляет всей сложностью за нас
# Думайте об этом как о предварительно созданном рабочем процессе: загрузить модель → обработать вход → сгенерировать выход
from transformers import pipeline, logging
logging.set_verbosity_error()  # Показывать только реальные ошибки, а не информационные сообщения

print('Установка завершена')

## Быстрый результат: Генерация текста с GPT-2
После ячейки «Настройка» запустите следующую ячейку. Вы должны увидеть начало `The secret to building AI is` и непустое продолжение. Конкретные слова могут меняться при каждом запуске.

In [ ]:
# Загрузить модель генерации текста GPT-2 (124M параметров)
generator = pipeline('text-generation', model='gpt2')

# Сгенерировать текст из начального промпта
result = generator(
    'The secret to building AI is',  # Начальный текст
    max_new_tokens=20,                # Добавить не более 20 новых токенов
    do_sample=True,                   # Выбирать токены случайно с учётом их вероятностей
    pad_token_id=50256                # Техническое: избегает предупреждения
)

# Извлечь и вывести сгенерированный текст
print(result[0]['generated_text'])

## Строки и базовые операции
Работа с текстом, длинами и срезами.

**Совет по Python: f-строки** позволяют вставлять значения переменных в текст. Буква `f` перед строкой означает "formatted" (форматированная), а фигурные скобки `{}` отмечают места для вставки значений:
```python
name = "GPT"
print(f"Hello, {name}!")  # → Hello, GPT!
```

In [ ]:
# Пример вывода из модели генерации текста
output = 'The secret to building AI is understanding how machines learn from data'
prompt = 'The secret to building AI is'

# Базовые операции со строками
print(len(output))  # Длина в символах
print(type(output))  # Подтвердить, что это строка

# Разделить на слова (список строк)
words = output.split()
print(words)

# Цепочка методов: вызов нескольких операций последовательно
# Это эквивалентно двум строкам:
#   lowercase_text = output.lower()
#   words = lowercase_text.split()
words = output.lower().split()
print(words)

# Извлечь только сгенерированную часть (срез от конца промпта)
generated = output[len(prompt):]
print(f'Сгенерировано: {generated.strip()}')  # .strip() удаляет начальные и конечные пробелы
print(f'Количество слов: {len(generated.split())}')

## Числа и форматирование
Базовые числовые значения и f-строки.

In [ ]:
# Python использует подчеркивания для читаемости в больших числах
num_parameters = 124_000_000  # 124 миллиона (размер GPT-2)
learning_rate = 0.0001        # Небольшой размер шага для обучения
vocab_size = 50257            # Размер словаря GPT-2

# Трюки форматирования f-строк:
print(f'GPT-2 имеет {num_parameters:,} параметров')   # :, добавляет разделители запятыми
print(f'Скорость обучения {learning_rate:.2e}')       # :.2e = научная нотация
print(f'Половина словаря: {vocab_size // 2}')         # Вы можете вычислять внутри {}!

## Построение маленького словаря и токенизатора
От игрушечных предложений к токенизатору на уровне слов.

In [ ]:
# Примеры текстовых данных (на которых вы бы обучали модель)
texts = [
    'The secret to building AI is',
    'The key to machine learning is data',
    'AI systems learn from examples'
]

# Собрать все слова из всех текстов
all_words = []
for text in texts:
    words = text.lower().split()  # Нормализовать к нижнему регистру
    # extend() добавляет каждый элемент индивидуально в список
    # (append() добавил бы весь список как ОДИН элемент)
    all_words.extend(words)

print(all_words)

# Примеры срезов списка
print(all_words[0], all_words[-1], all_words[:3])  # Первый, последний, первые 3

# Построить словарь: сопоставить каждое уникальное слово с числом
vocab = {'<PAD>': 0, '<UNK>': 1}  # Специальные токены сначала (зарезервированные ID)

for word in all_words:
    if word not in vocab:
        # len(vocab) дает "следующий доступный ID"
        # Если vocab имеет 2 элемента (ID 0 и 1), len(vocab)=2 становится следующим ID
        vocab[word] = len(vocab)

print(f'Размер словаря: {len(vocab)}')
print(vocab)

# Искать слова в словаре
print(vocab['the'], vocab['ai'])  # Возвращает их ID

## Сравнение с токенизатором GPT-2
Показать, чем отличается профессиональный токенизатор.

**Советы по Python:**
- `dict.get(key, default)` возвращает значение для `key`, если оно существует, иначе возвращает `default`. Безопаснее, чем `dict[key]`, который падает, если ключ отсутствует.
- **Генератор списка (list comprehension)** строит список за одно выражение. `[expr for item in collection]` эквивалентно циклу `for`, который добавляет элементы в список.

In [ ]:
# Загрузить настоящий токенизатор GPT-2
from transformers import GPT2Tokenizer
real_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Сравнить размеры словарей
print(f'Наш словарь: {len(vocab)} слов')
print(f'Словарь GPT-2: {len(real_tok)} токенов')

# Проверить обработку неизвестных слов
# .get(key, default) возвращает default, если ключ не найден (вместо падения)
word = 'neural'
print(f"'{word}' → {vocab.get(word, vocab['<UNK>'])}")  # Возвращает ID <UNK> (1)

# Токенизировать предложение с нашим словарем
sentence = 'The neural network learns'

# Длинная версия (явный цикл):
token_ids = []
for word in sentence.lower().split():
    token_id = vocab.get(word, vocab['<UNK>'])  # Получить ID или <UNK> если неизвестно
    token_ids.append(token_id)
    print(f'  {word} → {token_id}')

print(f'ID токенов (цикл): {token_ids}')

# Короткая версия: генератор списка даёт тот же результат.
token_ids = [vocab.get(w, vocab['<UNK>']) for w in sentence.lower().split()]
print(f'ID токенов (генератор списка): {token_ids}')

## Вспомогательные функции токенизации и детокенизации
Обработать предложение туда и обратно.

In [ ]:
# Функция: Текст → ID токенов (кодирование)
def tokenize(text, vocab):
    words = text.lower().split()
    return [vocab.get(w, vocab['<UNK>']) for w in words]

# Функция: ID токенов → Текст (декодирование)
def detokenize(ids, vocab):
    # Создать обратное отображение (ID → слово)
    # Словарное включение: {new_key: new_val for key, val in dict.items()}
    # vocab.items() возвращает пары вроде ('the', 2), ('secret', 3), и т.д.
    # Мы переворачиваем их: (2, 'the'), (3, 'secret'), и т.д.
    id_to_word = {v: k for k, v in vocab.items()}
    return ' '.join(id_to_word.get(i, '<UNK>') for i in ids)

# Проверить обработку туда-обратно: текст → ID → текст
ids = tokenize('The secret to AI', vocab)
print(f'Закодировано: {ids}')
print(f'Декодировано: {detokenize(ids, vocab)}')

# Сравнить наш токенизатор с GPT-2
text = 'The secret to AI'
print(f'Наши токены:   {tokenize(text, vocab)}')
print(f'Токены GPT-2: {real_tok.encode(text)}')  # Отличаются! GPT-2 использует подслова, а не целые слова

## Минимальный класс токенизатора
Токенизатор с состоянием на уровне слов с методами fit/encode/decode.

**Основы классов Python:**
**Класс** это чертеж для создания объектов, которые объединяют данные и функции вместе.

- `class MyClass:` — определяет чертеж
- `__init__(self)` — выполняется при создании нового объекта (инициализирует его данные)
- `self` — относится к "этому конкретному объекту" (как "эта машина" против "машины вообще")
- Методы (функции внутри класса) автоматически получают `self` в качестве первого параметра

**Аналогия:** Класс это как чертеж автомобиля. `__init__` устанавливает начальные характеристики (цвет, размер двигателя). Когда вы строите машину по чертежу, `self` относится к ЭТОЙ конкретной машине.

In [ ]:
# Объектно-ориентированный токенизатор (класс объединяет данные + методы)
class SimpleTokenizer:
    def __init__(self):
        # Инициализировать словарь специальными токенами
        self.word_to_id = {'<PAD>': 0, '<UNK>': 1}
        self.id_to_word = {0: '<PAD>', 1: '<UNK>'}

    def fit(self, texts):
        """Построить словарь из обучающих текстов"""
        for text in texts:
            for word in text.lower().split():
                if word not in self.word_to_id:
                    idx = len(self.word_to_id)
                    self.word_to_id[word] = idx  # Добавить новое слово
                    self.id_to_word[idx] = word  # Обратное отображение

    def encode(self, text):
        """Преобразовать текст в ID токенов"""
        return [self.word_to_id.get(w, 1) for w in text.lower().split()]  # 1 = <UNK>

    def decode(self, ids):
        """Преобразовать ID токенов обратно в текст"""
        return ' '.join(self.id_to_word.get(i, '<UNK>') for i in ids)

    def __len__(self):
        """Вернуть размер словаря (позволяет использовать len(tok))"""
        return len(self.word_to_id)

# Создать и обучить токенизатор
tok = SimpleTokenizer()
tok.fit(texts)  # Выучить словарь из наших обучающих данных

print(f'Размер словаря: {len(tok)}')

# Проверить кодирование/декодирование
ids = tok.encode('The secret to AI')
print(f'Закодировано: {ids}')
print(f'Декодировано: {tok.decode(ids)}')

# Сравнить с GPT-2 еще раз
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
text = 'The secret to AI'
print(f'Ваш токенизатор:  {tok.encode(text)}')
print(f'Токенизатор GPT-2: {gpt2_tok.encode(text)}')  # GPT-2 использует кодирование парами байтов (BPE)

## Полный круг: Ваш токенизатор против GPT-2

Давайте сравним ваш самодельный токенизатор с реальным токенизатором GPT-2 еще один последний раз. Обратите внимание, что ID токенов GPT-2 имеют гораздо большие числа (у него 50 257 токенов!) и он использует **токенизацию подслов** (BPE) вместо целых слов.

In [ ]:
# Создать свежий токенизатор и сравнить с GPT-2
my_tok = SimpleTokenizer()
my_tok.fit(['The secret to building AI is understanding'])

# Загрузить токенизатор GPT-2
from transformers import GPT2Tokenizer
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Сравнить на одном и том же тексте
text = 'The secret to AI'
print(f'Ваш токенизатор:  {my_tok.encode(text)}')
print(f'Токенизатор GPT-2: {gpt2_tok.encode(text)}')

print(f'\nРазмер вашего словаря:  {len(my_tok)}')
print(f'Размер словаря GPT-2: {len(gpt2_tok)}')

## Что только что произошло?

Вы построили полный токенизатор с нуля! Вот что вы изучили:

1. **Строки** — манипуляция текстом с `.split()`, `.lower()`, срезами
2. **Словари** — отображения ключ-значение для поиска в словаре
3. **Списки** — упорядоченные коллекции для последовательностей токенов
4. **Функции** — переиспользуемые блоки кода (`def tokenize(...)`)
5. **Классы** — чертежи, которые объединяют данные и методы вместе

**Ключевое понимание:** Ваш токенизатор использует целые слова, поэтому неизвестные слова становятся `<UNK>`. GPT-2 использует **кодирование парами байтов (BPE)**, которое разбивает слова на подслова, поэтому он редко видит действительно неизвестные токены. Вы узнаете больше об этом в Главе 8!